# RoadGuard AI - training

Run this on a GPU (Kaggle: Settings → Accelerator → GPU T4/P100, or Colab: Runtime → Change runtime type → GPU).
It merges the datasets, fine-tunes YOLOv8 and evaluates it on the held-out test split.

In [ ]:
!nvidia-smi
!git clone https://github.com/Sahil-u07/roadguard-ai
%cd roadguard-ai
!pip install -q ultralytics

## 1. Datasets

Download RAD, Indian Roads, Humps/Bumps/Potholes and HighRPD (links in the README) and unzip each into `raw/`
using the folder names in `sources.yaml`. Then fill in each source's class `map` in `sources.yaml`.

On Colab you can keep the zips on Google Drive:
```python
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p raw && unzip -q /content/drive/MyDrive/roadguard/RAD.zip -d raw/RAD
```

In [ ]:
!python merge_datasets.py sources.yaml datasets/roadguard

## 2. Train

We start from `yolov8n.pt` (COCO-pretrained) and fine-tune it. Transfer learning needs far less data and time
than training from scratch. Switch to `yolov8s.pt` for higher accuracy if you have the GPU time.
If the session disconnects, resume with `YOLO(path_to_last_pt).train(resume=True)`, where `last.pt` sits next to the `best.pt` printed below.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data="datasets/roadguard/data.yaml", epochs=50, imgsz=640, batch=16, patience=15,
            project="runs", name="roadguard", exist_ok=True)
best_path = model.trainer.best  # Ultralytics picks the run folder, so ask it where best.pt went
print(best_path)

## 3. Evaluate on the test split

The test split is never seen during training or model selection, so these are the numbers to report.
Ultralytics also saves the confusion matrix and PR/F1 curves next to `metrics.json` in the folder printed below.

In [ ]:
import json

best = YOLO(best_path)
m = best.val(data="datasets/roadguard/data.yaml", split="test", project="runs", name="test", exist_ok=True)

p, r = m.box.mp, m.box.mr
metrics = {"precision": p, "recall": r, "f1": 2 * p * r / (p + r), "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "per_class": {best.names[int(c)]: dict(zip(["precision", "recall", "mAP50", "mAP50-95"], m.box.class_result(i)))
                         for i, c in enumerate(m.box.ap_class_index)}}
print(json.dumps(metrics, indent=2, default=float))
json.dump(metrics, open(m.save_dir / "metrics.json", "w"), indent=2, default=float)
test_dir = m.save_dir

## 4. Save the model for the app

Copy `best.pt` into `weights/` and commit it so the Streamlit app can load it.

In [ ]:
import shutil
from pathlib import Path

Path("weights").mkdir(exist_ok=True)
shutil.copy(best_path, "weights/best.pt")
!zip -qr results.zip {model.trainer.save_dir} {test_dir} weights/best.pt
# Kaggle: download results.zip from the Output panel. Colab:
# from google.colab import files; files.download("results.zip")